# B3RK — SG Multiplicity (K replacing G2)

## Purpose

Roadmap step 6. Test whether SG's geometric multiplicity estimator, **G2** (2-cluster k-means on Marigold depth *values* — depth layers), can be improved by the K estimators developed in B3GK, inside SG's own frozen architecture:

| Variant | Multiplicity estimator on refinable recovery components | Prior |
|---|---|---|
| **SG_v1.1 (reference)** | G2 — k-means on depth values + G2 retention (v1.1 counting) | Depth layers |
| **SG-K3** | K3 — marker watershed on the depth-gradient magnitude | Depth edges |
| **SG-K2** | K2 — marker watershed on the distance transform | Shape (non-depth control) |

This also answers the open question from the first review: does SG's dense-scene gain come from **depth**, and if so from depth **layers** or depth **edges**?

## What stays frozen (SG_v1.1)

B0 anchor; A3c → C1 components; C1 recovery (centroid outside every B0 mask); split candidates (area percentile ≥ 0.75); Stage-1 gate (candidate area fraction ≥ 0.85) and Stage-2 gate (≥ 5.0 G2 extra units per refinable component, computed on SG_v1 extras); refinable set.

## Pre-declared K rule inside SG

For each refinable recovery component, only when Stage 2 is ON:
- **region** = component pixels **not covered by any B0 mask** (the intent of the v1.1 double-counting fix);
- **m** = K segments in the region (≥ 1 if the region is non-empty);
- **extra units** = max(m − 1, 0) (the component's +1 is already in S);
- **A_ref** = median B0 mask area (fallback: median C1 component area if the image has no B0 masks); K2/K3 keep segments ≥ 0.25 × A_ref; K2 h-maxima h = 1 px; K3 h = 0.1 × (p95 − p5) of the region's gradient (identical to B3GK).

Count = S + Σ extra units (weight 1.0, no cap). Implemented as `multiplicity.sg_k_count`.

## Pre-declared decision rule

**SG_v2 is frozen only if a variant improves on SG_v1.1 on both dev100 MAE and stratum-reweighted MAE.** Otherwise SG_v1.1 remains the SG candidate. dev100 only; held-out cache untouched.

## Sections

| Section | Content | GPU |
|---|---|---|
| 1 | Setup | No |
| 2 | Reproduce SG_v1.1 from frozen caches (100/100) | No |
| 3 | SG-K3 and SG-K2 | No |
| 4 | Comparison (incl. GPV_R4_v1, GPV_K_v1) and decision | No |

# 1. Setup

In [1]:
# 1.0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 1.1 — Paths, repo modules, references
# ============================================================

import importlib
import json
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"

DEV100_CSV = RESULTS_ROOT / "B1_experiments" / "subsets" / "FSC147_val_dev_100.csv"
B3_REFERENCE_FILE = (RESULTS_ROOT / "B3_B0_B2_M_Geometry_Integration" / "tables"
                     / "B3_step3F_frozen_architectures_per_image.csv")
SG_V11_CONFIG = RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "SG_v1.1_frozen_config.json"
SG_V11_PER_IMAGE = (RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "section4_SG_v1_1"
                    / "B3R_s4_SG_v1_1_per_image.csv")
GPV_FROZEN_COUNTS = (RESULTS_ROOT / "B3GK_GPV_Multiplicity_and_Prompts" / "frozen"
                     / "GPV_frozen_dev100_reference_counts.csv")

B3RK_DIR = RESULTS_ROOT / "B3RK_SG_Multiplicity"
B3RK_DIR.mkdir(parents=True, exist_ok=True)
for p in [DEV100_CSV, B3_REFERENCE_FILE, SG_V11_CONFIG, SG_V11_PER_IMAGE, GPV_FROZEN_COUNTS]:
    assert p.exists(), f"Missing: {p}"

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import src.pipeline.io_cache as io_cache
import src.pipeline.sg as sg
import src.pipeline.multiplicity as K
for m in (io_cache, sg, K):
    importlib.reload(m)
assert hasattr(K, "sg_k_count"), "Old multiplicity.py — commit the B3RK version."

SG_V11_CFG = sg.SGConfig.from_frozen_json(SG_V11_CONFIG)
assert SG_V11_CFG.variant == "v1.1"

dev100_df = pd.read_csv(DEV100_CSV).sort_values("dataset_index").reset_index(drop=True)
STRATA = ["S1_0_20", "S2_20_40", "S3_40_60", "S4_60_80", "S5_80_95", "S6_95_100"]

ref = (pd.read_csv(B3_REFERENCE_FILE)[["dataset_index", "B0_count", "S_count"]])
v11 = (pd.read_csv(SG_V11_PER_IMAGE)[["dataset_index", "SG_count", "stage2_on"]]
       .rename(columns={"SG_count": "SG_v1_1_count"}))
gpv = pd.read_csv(GPV_FROZEN_COUNTS)[["dataset_index", "GPV_R4_v1_count", "GPV_K_v1_count"]]
reference_df = (dev100_df.merge(ref, on="dataset_index").merge(v11, on="dataset_index")
                .merge(gpv, on="dataset_index"))
assert len(reference_df) == 100

print("Repo commit :", REPO_COMMIT)
print("SG config   :", SG_V11_CFG.to_dict())
print("Output      :", B3RK_DIR)

Repo commit : 3d0b35302299e43a9938a6e4bab1ef98048c320b
SG config   : {'candidate_quantile': 0.75, 'satellite_parent_fraction': 0.001, 'stage1_threshold': 0.85, 'stage2_threshold': 5.0, 'correction_weight': 1.0, 'kmeans_k': 2, 'kmeans_random_state': 0, 'kmeans_n_init': 10, 'variant': 'v1.1'}
Output      : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3RK_SG_Multiplicity


# 2. Reproduce SG_v1.1 (safety check)

Recompute SG_v1.1 from the frozen dev100 caches with `sg.compute_sg_image` (variant v1.1). Must match B3R Section 4 on 100/100 images before anything is changed.

In [3]:
# ============================================================
# Step 2.1 — SG_v1.1 from frozen caches
# ============================================================

CACHE = {}
repro = []
t0 = time.time()
for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="SG_v1.1"):
    di = row.dataset_index
    p = io_cache.cache_paths(di, row.image_id)
    b0_masks, _ = io_cache.load_b0_masks(p["b0_masks"])
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    depth = io_cache.load_depth(p["depth"])
    CACHE[di] = (b0_masks, a3c, depth)
    s, _ = sg.compute_sg_image(b0_masks, a3c, depth, SG_V11_CFG)
    repro.append({"dataset_index": di, "SG_v1_1_rebuilt": s["SG_count"],
                  "stage2_rebuilt": s["stage2_on"]})
repro_df = pd.DataFrame(repro).sort_values("dataset_index").reset_index(drop=True)

ok = int(np.isclose(repro_df["SG_v1_1_rebuilt"], reference_df["SG_v1_1_count"]).sum())
gate_ok = int((repro_df["stage2_rebuilt"].to_numpy() == reference_df["stage2_on"].to_numpy()).sum())
print(f"Runtime: {time.time() - t0:.0f} s")
print(f"SG_v1.1 count reproduced : {ok}/100  {'PASS' if ok == 100 else 'FAIL'}")
print(f"Stage-2 gate reproduced  : {gate_ok}/100  {'PASS' if gate_ok == 100 else 'FAIL'}")
REPRO_PASS = ok == 100 and gate_ok == 100

SG_v1.1:   0%|          | 0/100 [00:00<?, ?it/s]

Runtime: 283 s
SG_v1.1 count reproduced : 100/100  PASS
Stage-2 gate reproduced  : 100/100  PASS


# 3. SG-K3 and SG-K2

Same frozen SG_v1.1 pipeline; only the multiplicity estimator on refinable recovery components changes (pre-declared rule in the header). Images where the Stage-2 gate is OFF are unchanged by construction (count = S).

In [4]:
# ============================================================
# Step 3.1 — Run SG-K3 and SG-K2
# ============================================================

assert REPRO_PASS, "Reproduction failed — stop."
rows = []
t0 = time.time()
for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="SG-K"):
    di = row.dataset_index
    b0_masks, a3c, depth = CACHE[di]
    rec = {"dataset_index": di, "image_id": row.image_id, "category": row.category,
           "density_stratum": row.density_stratum, "gt_count": row.gt_count}
    for est in ["K3", "K2"]:
        r = K.sg_k_count(b0_masks, a3c, depth, SG_V11_CFG, est)
        rec[f"SG_{est}_count"] = r["SG_count"]
        rec[f"SG_{est}_extra"] = r["extra_units"]
        rec["S_count_k"] = r["S_count"]
        rec["stage2_on"] = r["stage2_on"]
        rec["n_refinable"] = r["n_refinable"]
        rec["G2_v1_1_count"] = r["G2_SG_count"]
    rows.append(rec)
sgk_df = pd.DataFrame(rows).sort_values("dataset_index").reset_index(drop=True)
sgk_df["SG_G2_extra"] = sgk_df["G2_v1_1_count"] - sgk_df["S_count_k"]
print(f"Runtime: {time.time() - t0:.0f} s")

# Consistency: gate-OFF images must equal S for every variant; G2 count == SG_v1.1
off = ~sgk_df["stage2_on"].to_numpy(bool)
print("G2 inside sg_k_count == SG_v1.1 :",
      f"{int(np.isclose(sgk_df['G2_v1_1_count'], reference_df['SG_v1_1_count']).sum())}/100")
for est in ["K3", "K2"]:
    same = int(np.isclose(sgk_df.loc[off, f"SG_{est}_count"], sgk_df.loc[off, "S_count_k"]).sum())
    print(f"SG-{est} == S on gate-OFF images : {same}/{int(off.sum())}")

SG-K:   0%|          | 0/100 [00:00<?, ?it/s]

Runtime: 52 s
G2 inside sg_k_count == SG_v1.1 : 100/100
SG-K3 == S on gate-OFF images : 84/84
SG-K2 == S on gate-OFF images : 84/84


In [5]:
# ============================================================
# Step 3.2 — Extra units by stratum (gate-ON images)
# ============================================================

on = sgk_df[sgk_df["stage2_on"]]
units = on.groupby("density_stratum")[["SG_G2_extra", "SG_K3_extra", "SG_K2_extra",
                                       "gt_count", "S_count_k"]].sum().reindex(STRATA)
units["GT_minus_S"] = units["gt_count"] - units["S_count_k"]
print(f"Gate-ON images: {len(on)}")
print("Extra units added per stratum (gate-ON images) vs the remaining deficit GT − S:")
display(units)

print("\nPer gate-ON image:")
display(on[["image_id", "category", "density_stratum", "gt_count", "S_count_k",
            "G2_v1_1_count", "SG_K3_count", "SG_K2_count", "n_refinable"]]
        .sort_values("gt_count", ascending=False))

Gate-ON images: 16
Extra units added per stratum (gate-ON images) vs the remaining deficit GT − S:


,SG_G2_extra,SG_K3_extra,SG_K2_extra,gt_count,S_count_k,GT_minus_S
density_stratum,,,,,,
S1_0_20,27.0,22,2,30,41,-11
S2_20_40,34.0,81,5,29,25,4
S3_40_60,31.0,43,21,80,47,33
S4_60_80,51.0,36,15,169,74,95
S5_80_95,31.0,6,1,113,70,43
S6_95_100,995.0,916,66,2281,367,1914



Per gate-ON image:


,image_id,category,density_stratum,gt_count,S_count_k,G2_v1_1_count,SG_K3_count,SG_K2_count,n_refinable
93,6969.jpg,birds,S6_95_100,949,53,289.0,481.0,76.0,14
32,975.jpg,birds,S6_95_100,718,107,726.0,520.0,132.0,27
92,6931.jpg,birds,S6_95_100,355,99,234.0,166.0,113.0,25
0,215.jpg,grapes,S6_95_100,259,108,113.0,116.0,112.0,1
24,838.jpg,books,S5_80_95,113,70,101.0,76.0,71.0,8
99,7060.jpg,pills,S4_60_80,65,25,47.0,59.0,34.0,4
78,5896.jpg,grapes,S4_60_80,55,44,65.0,46.0,50.0,3
83,6286.jpg,pills,S4_60_80,49,5,13.0,5.0,5.0,1
42,2840.jpg,chairs,S3_40_60,35,8,14.0,8.0,8.0,1
69,5101.jpg,chicken wings,S3_40_60,23,30,38.0,31.0,31.0,2


# 4. Comparison and decision

In [6]:
# ============================================================
# Step 4.1 — Metrics (dev100 + stratum-reweighted)
# ============================================================

STRATUM_WEIGHTS = {"S1_0_20": 0.20, "S2_20_40": 0.20, "S3_40_60": 0.20,
                   "S4_60_80": 0.20, "S5_80_95": 0.15, "S6_95_100": 0.05}

def summary(name, pred):
    gt = reference_df["gt_count"].to_numpy().astype(float)
    pred = np.asarray(pred, float)
    err = pred - gt
    st = reference_df["density_stratum"].to_numpy()
    out = {"system": name,
           "MAE": float(np.abs(err).mean()),
           "MAE_reweighted": float(sum(w * np.abs(err[st == s]).mean() for s, w in STRATUM_WEIGHTS.items())),
           "RMSE": float(np.sqrt((err ** 2).mean())),
           "NAE": float((np.abs(err) / gt).mean()),
           "mean_signed_error": float(err.mean()),
           "MAE_S1_S5": float(np.abs(err[st != "S6_95_100"]).mean()),
           "MAE_S6": float(np.abs(err[st == "S6_95_100"]).mean())}
    for s in STRATA:
        out[f"MAE_{s[:2]}"] = float(np.abs(err[st == s]).mean())
    return out

systems = [("B0", reference_df["B0_count"]), ("S", reference_df["S_count"]),
           ("SG_v1.1 (G2: depth layers)", reference_df["SG_v1_1_count"]),
           ("SG-K3 (depth edges)", sgk_df["SG_K3_count"]),
           ("SG-K2 (shape)", sgk_df["SG_K2_count"]),
           ("GPV_R4_v1", reference_df["GPV_R4_v1_count"]),
           ("GPV_K_v1", reference_df["GPV_K_v1_count"])]
metrics_b3rk = pd.DataFrame([summary(n, p) for n, p in systems])
display(metrics_b3rk.round(2))

,system,MAE,MAE_reweighted,RMSE,NAE,mean_signed_error,MAE_S1_S5,MAE_S6,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5
0,B0,80.69,52.65,270.23,0.56,-77.47,20.44,622.9,5.15,6.55,14.95,34.30,62.1
1,S,72.00,44.78,262.24,0.43,-65.32,14.57,588.9,4.55,5.60,8.95,24.15,44.6
2,SG_v1.1 (G2: depth layers),61.99,39.52,246.59,0.44,-53.63,14.32,491.0,5.70,6.80,8.60,22.60,41.5
3,SG-K3 (depth edges),63.62,40.96,243.57,0.50,-54.28,15.43,497.3,5.65,9.65,9.80,22.35,44.0
4,SG-K2 (shape),71.20,44.31,260.71,0.43,-64.22,14.41,582.3,4.65,5.85,8.70,23.40,44.5
5,GPV_R4_v1,69.91,42.31,263.05,0.42,-61.59,12.39,587.6,4.65,7.05,9.35,16.95,35.5
6,GPV_K_v1,54.32,37.06,151.10,0.72,-34.78,18.07,380.6,16.90,7.35,9.35,29.95,35.5


In [7]:
# ============================================================
# Step 4.2 — Pre-declared decision and save
# ============================================================

m = metrics_b3rk.set_index("system")
base = m.loc["SG_v1.1 (G2: depth layers)"]
decision_rows = []
for v in ["SG-K3 (depth edges)", "SG-K2 (shape)"]:
    better_mae = m.loc[v, "MAE"] < base["MAE"]
    better_rw = m.loc[v, "MAE_reweighted"] < base["MAE_reweighted"]
    decision_rows.append({"variant": v,
                          "dMAE_vs_SG_v1.1": m.loc[v, "MAE"] - base["MAE"],
                          "dMAE_reweighted_vs_SG_v1.1": m.loc[v, "MAE_reweighted"] - base["MAE_reweighted"],
                          "improves_both": bool(better_mae and better_rw)})
decision_df = pd.DataFrame(decision_rows)
display(decision_df.round(2))

winners = decision_df[decision_df["improves_both"]]
if len(winners):
    best = winners.sort_values("dMAE_reweighted_vs_SG_v1.1").iloc[0]["variant"]
    DECISION = f"Candidate for SG_v2: {best} (improves dev100 MAE and reweighted MAE)"
else:
    DECISION = "No variant improves both metrics — SG_v1.1 remains the SG candidate"
print("\nDECISION (pre-declared rule):", DECISION)

sgk_df.to_csv(B3RK_DIR / "B3RK_per_image.csv", index=False)
metrics_b3rk.to_csv(B3RK_DIR / "B3RK_metrics.csv", index=False)
units.to_csv(B3RK_DIR / "B3RK_extra_units_by_stratum.csv")
decision_df.to_csv(B3RK_DIR / "B3RK_decision.csv", index=False)
with open(B3RK_DIR / "B3RK_metadata.json", "w") as f:
    json.dump({
        "notebook": "B3RK_SG_Multiplicity",
        "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "repo_commit": REPO_COMMIT, "sg_config": SG_V11_CFG.to_dict(),
        "k_rule": {"region": "refinable recovery component minus B0 union (Stage 2 ON only)",
                   "multiplicity": "K segments (>=1 if region non-empty)",
                   "extra": "max(m - 1, 0)",
                   "A_ref": "median B0 mask area (fallback median C1 component area)",
                   "min_segment": "0.25 x A_ref",
                   "K2": "distance-transform watershed, h-maxima h=1",
                   "K3": "depth-gradient watershed, h-minima h=0.1*(p95-p5), Gaussian sigma 1"},
        "decision_rule": "freeze SG_v2 only if a variant improves both dev100 MAE and stratum-reweighted MAE",
        "decision": DECISION, "reproduction_pass": REPRO_PASS,
        "new_inference": False, "heldout_accessed": False, "test_set_accessed": False,
    }, f, indent=2)
print("Saved to:", B3RK_DIR)

,variant,dMAE_vs_SG_v1.1,dMAE_reweighted_vs_SG_v1.1,improves_both
0,SG-K3 (depth edges),1.63,1.44,False
1,SG-K2 (shape),9.21,4.80,False



DECISION (pre-declared rule): No variant improves both metrics — SG_v1.1 remains the SG candidate
Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3RK_SG_Multiplicity


## Section 4 decision

No K variant improves SG_v1.1 on both metrics (SG-K3: +1.63 MAE, +1.44 reweighted; SG-K2: +9.21, +4.80). **SG_v1.1 remains the SG candidate.**

- **SG's dense-scene gain comes from depth:** in gate-ON S6 images the shape control K2 adds 66 units vs 995 (G2) and 916 (K3); SG-K2 ≈ S without geometry (71.20 vs 72.00).
- **Inside SG, depth layers (G2) ≈ depth edges (K3)**; G2's small edge comes mostly from 975.jpg (G2 726 vs GT 718; K3 520).
- Contrast with GPV: on large merged SAM2 masks, edges ≫ layers/shape; on SG's compact A3c blobs, layers ≈ edges.

# 5. Cross-architecture oracle (diagnostic only)

## Purpose

Before designing any SG/GPV hybrid (roadmap step 7), measure the **ceiling** of routing between architectures: for each image, take whichever system's count is closer to GT. This uses GT, so it is **not a method** — it bounds what any GT-free hybrid could achieve.

## Pre-declared interpretation

- If the oracle gain over the best single system is **small** (< ~2–3 MAE on the stratum-reweighted estimate), no hybrid is worth building → skip step 7 and go to B4 with SG_v1.1, GPV_R4_v1, GPV_K_v1.
- If it is **large**, inspect which images drive it and whether a plausible GT-free signal could separate them.

## Pairs

SG_v1.1 vs GPV_K_v1 (main question); SG_v1.1 vs GPV_R4_v1; GPV_R4_v1 vs GPV_K_v1 (ceiling of Gate 1 itself); all three.

**Run order:** Section 1 only (CPU), then Section 5.

In [8]:
# ============================================================
# Step 5.1 — Oracle MAE for each pair / triple
# ============================================================

STRATUM_WEIGHTS = {"S1_0_20": 0.20, "S2_20_40": 0.20, "S3_40_60": 0.20,
                   "S4_60_80": 0.20, "S5_80_95": 0.15, "S6_95_100": 0.05}
gt = reference_df["gt_count"].to_numpy(float)
st = reference_df["density_stratum"].to_numpy()

SYS = {"SG_v1.1": reference_df["SG_v1_1_count"].to_numpy(float),
       "GPV_R4_v1": reference_df["GPV_R4_v1_count"].to_numpy(float),
       "GPV_K_v1": reference_df["GPV_K_v1_count"].to_numpy(float)}

def stats(abs_err, name):
    out = {"system": name, "MAE": abs_err.mean(),
           "MAE_reweighted": sum(w * abs_err[st == s].mean() for s, w in STRATUM_WEIGHTS.items()),
           "MAE_S1_S5": abs_err[st != "S6_95_100"].mean(),
           "MAE_S6": abs_err[st == "S6_95_100"].mean()}
    for s in STRATA:
        out[f"MAE_{s[:2]}"] = abs_err[st == s].mean()
    return out

rows = [stats(np.abs(p - gt), n) for n, p in SYS.items()]
ORACLES = {"oracle(SG_v1.1, GPV_K_v1)": ["SG_v1.1", "GPV_K_v1"],
           "oracle(SG_v1.1, GPV_R4_v1)": ["SG_v1.1", "GPV_R4_v1"],
           "oracle(GPV_R4_v1, GPV_K_v1)": ["GPV_R4_v1", "GPV_K_v1"],
           "oracle(all three)": ["SG_v1.1", "GPV_R4_v1", "GPV_K_v1"]}
choice = {}
for name, members in ORACLES.items():
    errs = np.stack([np.abs(SYS[m] - gt) for m in members])
    choice[name] = np.array(members)[errs.argmin(axis=0)]
    rows.append(stats(errs.min(axis=0), name))
oracle_df = pd.DataFrame(rows)
display(oracle_df.round(2))

best_single_rw = oracle_df.iloc[:3]["MAE_reweighted"].min()
best_single_mae = oracle_df.iloc[:3]["MAE"].min()
print(f"\nBest single system: MAE {best_single_mae:.2f} | reweighted {best_single_rw:.2f}")
for name in ORACLES:
    r = oracle_df.set_index("system").loc[name]
    print(f"{name:<30}: gain vs best single — MAE {best_single_mae - r['MAE']:.2f} | "
          f"reweighted {best_single_rw - r['MAE_reweighted']:.2f}")

,system,MAE,MAE_reweighted,MAE_S1_S5,MAE_S6,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5
0,SG_v1.1,61.99,39.52,14.32,491.0,5.70,6.80,8.60,22.60,41.5
1,GPV_R4_v1,69.91,42.31,12.39,587.6,4.65,7.05,9.35,16.95,35.5
2,GPV_K_v1,54.32,37.06,18.07,380.6,16.90,7.35,9.35,29.95,35.5
3,"oracle(SG_v1.1, GPV_K_v1)",40.28,26.78,11.14,302.5,3.70,4.55,6.15,19.45,32.6
4,"oracle(SG_v1.1, GPV_R4_v1)",57.59,35.10,10.39,482.4,3.70,4.25,6.15,16.35,32.6
5,"oracle(GPV_R4_v1, GPV_K_v1)",48.99,31.74,12.14,380.6,4.65,7.05,9.35,15.85,35.5
6,oracle(all three),39.42,25.92,10.19,302.5,3.70,4.25,6.15,15.45,32.6



Best single system: MAE 54.32 | reweighted 37.06
oracle(SG_v1.1, GPV_K_v1)     : gain vs best single — MAE 14.04 | reweighted 10.28
oracle(SG_v1.1, GPV_R4_v1)    : gain vs best single — MAE -3.27 | reweighted 1.96
oracle(GPV_R4_v1, GPV_K_v1)   : gain vs best single — MAE 5.33 | reweighted 5.33
oracle(all three)             : gain vs best single — MAE 14.90 | reweighted 11.14


In [9]:
# ============================================================
# Step 5.2 — Where does the main oracle (SG_v1.1 vs GPV_K_v1) gain?
# ============================================================

main = "oracle(SG_v1.1, GPV_K_v1)"
e_sg = np.abs(SYS["SG_v1.1"] - gt)
e_k = np.abs(SYS["GPV_K_v1"] - gt)
gain = e_k - np.minimum(e_sg, e_k)          # error removed by switching to SG when SG is better

detail = reference_df[["image_id", "category", "density_stratum", "gt_count"]].copy()
detail["SG_v1.1"] = SYS["SG_v1.1"]
detail["GPV_K_v1"] = SYS["GPV_K_v1"]
detail["GPV_R4_v1"] = SYS["GPV_R4_v1"]
detail["oracle_picks"] = choice[main]
detail["gain_vs_GPV_K"] = gain

print("Images where SG_v1.1 is closer than GPV_K_v1 (sorted by gain):")
display(detail[detail["gain_vs_GPV_K"] > 0].sort_values("gain_vs_GPV_K", ascending=False))

g_tot = gain.sum()
print(f"\nTotal error removable by the oracle: {g_tot:.0f} count units "
      f"({g_tot / len(gt):.2f} MAE points)")
top3 = np.sort(gain)[::-1][:3].sum()
print(f"Share from the top 3 images: {top3 / g_tot:.0%}" if g_tot else "")

print("\nOracle picks by stratum:")
display(pd.crosstab(detail["density_stratum"], detail["oracle_picks"]).reindex(STRATA))

Images where SG_v1.1 is closer than GPV_K_v1 (sorted by gain):


,image_id,category,density_stratum,gt_count,SG_v1.1,GPV_K_v1,GPV_R4_v1,oracle_picks,gain_vs_GPV_K
32,975.jpg,birds,S6_95_100,718,726.0,71.0,71.0,SG_v1.1,639.0
13,737.jpg,ants,S1_0_20,9,17.0,265.0,20.0,SG_v1.1,248.0
14,745.jpg,ants,S4_60_80,60,12.0,281.0,48.0,SG_v1.1,173.0
92,6931.jpg,birds,S6_95_100,355,234.0,97.0,97.0,SG_v1.1,137.0
1,220.jpg,grapes,S4_60_80,59,52.0,94.0,58.0,SG_v1.1,28.0
98,7100.jpg,milk cartons,S3_40_60,27,48.0,70.0,70.0,SG_v1.1,22.0
86,6896.jpg,books,S2_20_40,15,19.0,40.0,40.0,SG_v1.1,21.0
50,3561.jpg,chicken wings,S2_20_40,19,20.0,32.0,32.0,SG_v1.1,12.0
49,3553.jpg,chicken wings,S2_20_40,16,20.0,31.0,31.0,SG_v1.1,11.0
48,3543.jpg,chicken wings,S3_40_60,30,40.0,50.0,50.0,SG_v1.1,10.0



Total error removable by the oracle: 1404 count units (14.04 MAE points)
Share from the top 3 images: 75%

Oracle picks by stratum:


oracle_picks,GPV_K_v1,SG_v1.1
density_stratum,,
S1_0_20,9,11
S2_20_40,9,11
S3_40_60,8,12
S4_60_80,11,9
S5_80_95,4,6
S6_95_100,7,3


In [10]:
# ============================================================
# Step 5.3 — Save
# ============================================================

S5_DIR = B3RK_DIR / "section5_oracle"
S5_DIR.mkdir(parents=True, exist_ok=True)
oracle_df.to_csv(S5_DIR / "B3RK_s5_oracle_metrics.csv", index=False)
detail.to_csv(S5_DIR / "B3RK_s5_oracle_per_image.csv", index=False)
with open(S5_DIR / "B3RK_s5_metadata.json", "w") as f:
    json.dump({
        "notebook": "B3RK_SG_Multiplicity", "section": "5 — cross-architecture oracle",
        "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "purpose": "GT-based ceiling of routing between SG and GPV (diagnostic, not a method)",
        "pre_declared_interpretation": "gain < ~2-3 reweighted MAE -> skip hybrids, go to B4",
        "systems": list(SYS), "heldout_accessed": False, "test_set_accessed": False,
    }, f, indent=2)
print("Saved to:", S5_DIR)

Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3RK_SG_Multiplicity/section5_oracle


# B3RK conclusion

**SG multiplicity (Sections 2–4):** SG_v1.1 reproduced 100/100. No K variant improves SG_v1.1 on both dev100 MAE and stratum-reweighted MAE → **SG_v1.1 remains the SG candidate** (no SG_v2).
- **SG's dense-scene gain comes from depth:** the shape control (K2) adds 66 units in gate-ON S6 images vs 995 (G2) and 916 (K3); SG-K2 ≈ S without geometry.
- **Inside SG, depth layers (G2) ≈ depth edges (K3)**; G2's small edge comes mostly from 975.jpg.
- Contrast with GPV: on large merged SAM2 masks, edges ≫ layers/shape; on compact A3c blobs, layers ≈ edges.

**Cross-architecture oracle (Section 5, diagnostic):**
- Ceiling gain over the best single system (GPV_K_v1): 10.28 reweighted MAE for SG_v1.1 + GPV_K_v1; 11.14 for all three.
- Much of it is noise-picking between two noisy systems (oracle picks SG in ~half of S1–S3 images for gains of 1–5 units).
- 75% of the removable error comes from 3 images: 975 and 6931 (SG splits flocks GPV's Gate 1 misses) and 737 / 745 (ants: Gate 1 false alarms). The ants cases share Gate 1's pattern with 935.jpg (GPV_K's largest success), so no GT-free gate rule can separate them.

**Next:** hybrid architectures in `B3H_Hybrid_SG_GPV.ipynb`:
- **H1** — GPV-anchored SG (GPV_R4 masks minus rule-A merged masks as the anchor; C1 recovery + gated G2 on what remains).
- **H2** — routing: SG_v1.1 where SG's gate fires and GPV's Gate 1 does not; GPV_K_v1 otherwise.